In [138]:
# --- STEP 0: Install deps (if not already) ---
import pandas as pd
import numpy as np
from scipy.signal import resample
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from scipy.fft import fft
import requests

In [139]:
# API URL to fetch the latest data
file_path = 'https://io-t-ppg-bp-monitor-backend.vercel.app/api/finalData/latest'

def load_and_convert_to_csv(file_path, csv_filename='data.csv'):
    # Send GET request to the API
    response = requests.get(file_path.strip())
    
    # Check if the request was successful (status code 200)
    response.raise_for_status()  # Will raise an exception if status is not 200
    
    # Parse the JSON data
    data = response.json()
    
    # Normalize the JSON data to create a DataFrame
    df = pd.json_normalize(data)
    
    # Save the DataFrame to a CSV file
    df.to_csv(csv_filename, index=False)
    print(f"Data saved to {csv_filename}")
    
    return df

# Example usage
df = load_and_convert_to_csv(file_path)


Data saved to data.csv


In [140]:
display(df.head())

,heartRate,spo2,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP,ir,red
0,34,99,22,Male,167.64,69,24.55,114,75,"[53517, 53520, 53513, 53504, 53451, 53451, 534...","[43090, 43104, 43074, 43036, 42990, 43030, 430..."


In [141]:
from sklearn.preprocessing import LabelEncoder

# Initialize the LabelEncoder
le = LabelEncoder()

# Label encode the gender column
df['userGender'] = le.fit_transform(df['userGender'])

# Check the result
print(df['userGender'].head())

# Optionally, check the mapping of labels to values
print(le.classes_)  # This will print the unique categories mapped to 0, 1, 2, etc.


0    0
Name: userGender, dtype: int64
['Male']


In [142]:
display(df.head())

,heartRate,spo2,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP,ir,red
0,34,99,22,0,167.64,69,24.55,114,75,"[53517, 53520, 53513, 53504, 53451, 53451, 534...","[43090, 43104, 43074, 43036, 42990, 43030, 430..."


In [143]:
def convert_arrays_to_columns(df, array_columns):
    for col in array_columns:
        # Convert each element in the array column to a separate column
        array_expanded = pd.DataFrame(df[col].tolist(), index=df.index)

        #i = 2 # Initialize a counter for duplicate column names
        
        # Rename the new columns to match the original column with index
        array_expanded.columns = [f"{col}_{i}" for i in range(array_expanded.shape[1])]
        
        # Concatenate the expanded columns with the original DataFrame
        df = pd.concat([df, array_expanded], axis=1)
        
        # Optionally, drop the original array column
        df = df.drop(columns=[col])
    
    return df

# Example usage
df = convert_arrays_to_columns(df, ['ir', 'red'])
display(df.head())


,heartRate,spo2,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP,ir_0,...,red_159,red_160,red_161,red_162,red_163,red_164,red_165,red_166,red_167,red_168
0,34,99,22,0,167.64,69,24.55,114,75,53517,...,42345,42337,42318,42308,42291,42263,42282,42277,42262,42264


In [144]:

# --- STEP 3: Identify IR and RED columns ---
ir_cols = [c for c in df.columns if c.startswith("ir_")]
red_cols = [c for c in df.columns if c.startswith("red_")]

In [145]:
# --- STEP 4: Function to resample waveform to fixed length ---
def resample_to_fixed_length(row, cols, target_len=200):
    arr = row[cols].to_numpy(dtype=float)
    # Remove NaNs before resampling (shorten series if needed)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return np.full(target_len, np.nan)  # empty signal
    return resample(arr, target_len)


In [146]:
# --- STEP 5: Apply resampling to IR and RED ---
target_length = 200
ir_resampled = np.vstack(df.apply(lambda row: resample_to_fixed_length(row, ir_cols, target_length), axis=1))
red_resampled = np.vstack(df.apply(lambda row: resample_to_fixed_length(row, red_cols, target_length), axis=1))

In [147]:
# --- STEP 6: Save resampled signals into DataFrame ---
ir_df = pd.DataFrame(ir_resampled, columns=[f"ir_{i}" for i in range(target_length)])
red_df = pd.DataFrame(red_resampled, columns=[f"red_{i}" for i in range(target_length)])

In [148]:
display(ir_df.head())

,ir_0,ir_1,ir_2,ir_3,ir_4,ir_5,ir_6,ir_7,ir_8,ir_9,...,ir_190,ir_191,ir_192,ir_193,ir_194,ir_195,ir_196,ir_197,ir_198,ir_199
0,53517.0,53554.36034,53473.967804,53549.633638,53457.589865,53460.878064,53447.660325,53457.970774,53489.617251,53472.448274,...,52936.98981,52894.783511,52920.749853,52894.879627,52839.790424,52873.076436,52841.91613,52908.786849,52829.668395,52968.099016


In [149]:
# --- STEP 7: Merge back with meta + labels ---
keep_cols = ['spo2', 'heartRate', 'userAge', 'userGender', 'userHeight', 'userWeight', 'userBmi']
df_clean = pd.concat([df[keep_cols].reset_index(drop=True), ir_df, red_df], axis=1)

In [150]:
display(df_clean.head())

,spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,ir_0,ir_1,ir_2,...,red_190,red_191,red_192,red_193,red_194,red_195,red_196,red_197,red_198,red_199
0,99,34,22,0,167.64,69,24.55,53517.0,53554.36034,53473.967804,...,42341.679599,42295.963102,42319.929754,42282.070938,42257.607365,42297.746054,42245.37776,42325.075741,42184.062466,42366.026346


In [151]:
# --- Step 1: Extract the relevant columns for IR and RED signals ---
ir_cols = [c for c in df_clean.columns if c.startswith("ir_")]
red_cols = [c for c in df_clean.columns if c.startswith("red_")]

In [152]:
# --- Function to extract time-domain and frequency-domain features from a signal ---
def extract_features(signal):
    # Ensure the signal is a float numpy array
    signal = np.asarray(signal, dtype=float)

    # Handle cases with all NaN values after conversion
    if np.all(np.isnan(signal)):
        # Return a list of NaNs for all features if the signal is all NaNs
        return [np.nan] * 14 # 14 is the number of features being extracted

    # Time-domain features
    mean_signal = np.nanmean(signal) # Use nanmean to ignore NaNs
    median_signal = np.nanmedian(signal) # Use nanmedian to ignore NaNs
    std_signal = np.nanstd(signal) # Use nanstd to ignore NaNs
    min_signal = np.nanmin(signal) if not np.all(np.isnan(signal)) else np.nan # Handle min/max on all NaNs
    max_signal = np.nanmax(signal) if not np.all(np.isnan(signal)) else np.nan # Handle min/max on all NaNs
    range_signal = max_signal - min_signal if not np.all(np.isnan(signal)) else np.nan # Handle range on all NaNs

    # SciPy stats functions handle NaNs with nan_policy='omit' or 'propagate'
    # Let's ensure we handle potential NaNs explicitly or rely on default nan_policy
    skewness_signal = skew(signal, nan_policy='omit') if len(signal[~np.isnan(signal)]) > 1 else np.nan # Need at least 2 non-NaN for skew
    kurtosis_signal = kurtosis(signal, nan_policy='omit') if len(signal[~np.isnan(signal)]) > 3 else np.nan # Need at least 4 non-NaN for kurtosis


    # Peak features (find peaks in the signal) - find_peaks requires non-NaN input
    non_nan_signal = signal[~np.isnan(signal)]
    peaks, _ = find_peaks(non_nan_signal)
    num_peaks = len(peaks)
    mean_peak_amplitude = np.nanmean(non_nan_signal[peaks]) if num_peaks > 0 else np.nan # Use nanmean

    # Frequency-domain features using FFT - fft requires non-NaN input
    if len(non_nan_signal) == 0:
         return [np.nan] * 14 # Return NaNs if no non-NaN data for FFT

    fft_values = fft(non_nan_signal)
    fft_freq = np.fft.fftfreq(len(non_nan_signal))
    fft_magnitude = np.abs(fft_values)

    # Extract dominant frequency (proxy for heart rate)
    # Need to handle case where all magnitudes are zero (e.g., constant signal)
    if np.all(fft_magnitude[1:] == 0):
         dominant_freq = np.nan
    else:
        dominant_freq = np.abs(fft_freq[np.argmax(fft_magnitude[1:]) + 1])  # Skip zero frequency (DC component)


    # Total spectral energy
    total_energy = np.sum(fft_magnitude**2)



    return [
        mean_signal, median_signal, std_signal, min_signal, max_signal, range_signal,
        skewness_signal, kurtosis_signal, num_peaks, mean_peak_amplitude,
        dominant_freq, total_energy
    ]

In [ ]:
# --- Extract features for both IR and Red signals ---
def extract_features_from_df(df_clean, ir_columns):
    features = []
    for i, row in df_clean.iterrows():
        # Extract IR and Red signal features
        ir_signal = row[ir_columns].values

        ir_features = extract_features(ir_signal)

        # Combine IR and Red features into one set
        combined_features = ir_features

        # Append demographics
        demographics = [
            row['userAge'], row['userGender'], row['userHeight'], row['userWeight'],
            row['userBmi'], row['heartRate']  # Assuming these columns are present
        ]

        features.append(combined_features + demographics)

    return np.array(features)


In [ ]:
# --- Prepare feature matrix and labels ---
X = extract_features_from_df(df_clean, ir_cols)


In [158]:
# --- Create a DataFrame for the features ---
columns = [
    'mean', 'median', 'std', 'min', 'max', 'range', 'skewness', 'kurtosis', 'num_peaks', 'mean_peak_amplitude', 'dominant_freq',
     'total_energy', 'userAge', 'userGender', 'userHeight', 'userWeight', 'heartRate', 'userBmi'
]

In [159]:
X_df = pd.DataFrame(X, columns=columns)

In [160]:
display(X_df.head())

,mean,median,std,min,max,range,skewness,kurtosis,num_peaks,mean_peak_amplitude,dominant_freq,total_energy,userAge,userGender,userHeight,userWeight,heartRate,userBmi
0,53238.14201183433,53315.52650424013,170.63862974958852,52829.66839504023,53554.36033967929,724.6919446390602,-0.7730193397647515,-0.45061854555896597,53,53247.857837751675,0.005,113373155296568.05,22,0,167.64000000000001,69,24.55,34
